# Predição com LSTM e GRU usando 3 datasets do Yahoo Finance


O intuito é **comparar** **LSTM** e **GRU**, para prever o preço de fechamento de uma ação.

Foram usados **no mínimo 3 datasets do Yahoo Finance**:
- `PETR4.SA`
- `VALE3.SA`
- `ITUB4.SA`

O **alvo da predição** é o fechamento do primeiro ticker da lista, por padrão **`PETR4.SA`**.

## Blocos
1. instalação das bibliotecas
2. importações
3. configuração do experimento
4. utilitários
5. preparação dos dados
6. definição dos modelos
7. treinamento e avaliação
8. execução final



## A lógica central é:
- baixar dados históricos do Yahoo Finance
- enriquecer os dados com indicadores técnicos
- transformar a série temporal em janelas sequenciais
- treinar dois modelos diferentes
- comparar qual deles prevê melhor o fechamento do ativo alvo

In [1]:
!pip install -q yfinance pandas scikit-learn torch matplotlib

In [2]:
import warnings
warnings.filterwarnings('ignore')

import math
import os
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, Dataset
import yfinance as yf

## Bloco 3, Configuração do experimento

### O que este bloco faz
Define todos os parâmetros principais do projeto:
- tickers usados
- ativo alvo
- período histórico
- tamanho da janela temporal
- quantidade de épocas
- taxa de aprendizado
- divisão treino e teste
- dispositivo de processamento

### Intuito
Centralizar as decisões do experimento em um único lugar.
Assim, fica fácil alterar o estudo sem mexer em todo o código.

In [3]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

TICKERS = ['PETR4.SA', 'VALE3.SA', 'ITUB4.SA']
TARGET_TICKER = TICKERS[0]
START_DATE = '2018-01-01'
END_DATE = None
SEQ_LEN = 60
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-3
TRAIN_SPLIT = 0.8
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = 'outputs_lstm_gru'

print('Tickers usados:', TICKERS)
print('Ticker alvo:', TARGET_TICKER)
print('Dispositivo:', DEVICE)

Tickers usados: ['PETR4.SA', 'VALE3.SA', 'ITUB4.SA']
Ticker alvo: PETR4.SA
Dispositivo: cpu


## Bloco 4, Utilitários gerais

### O que este bloco faz
Cria funções auxiliares que facilitam o restante do notebook.

### Intuito
Evitar repetição de código e deixar a lógica mais organizada.

### Explicação dos utilitários
- `ensure_dir`: cria a pasta onde os arquivos finais serão salvos
- `inverse_target`: desfaz a normalização do valor previsto para voltar ao preço real
- `evaluate_predictions`: calcula métricas para comparar desempenho dos modelos

In [4]:
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def inverse_target(scaled_values: np.ndarray, scaler_target: MinMaxScaler) -> np.ndarray:
    return scaler_target.inverse_transform(scaled_values.reshape(-1, 1)).flatten()


def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    return {'RMSE': float(rmse), 'MAE': float(mae), 'MAPE_%': float(mape)}

## Bloco 5, Utilitários de engenharia de atributos

### O que este bloco faz
Cria indicadores técnicos a partir do preço de fechamento de cada ativo.

### Intuito
Dar mais contexto ao modelo.
Em vez de enxergar apenas o preço bruto, a rede passa a receber sinais que ajudam a interpretar tendência e força do movimento.

### Indicadores adicionados
- **SMA**: média móvel simples
- **EMA**: média móvel exponencial
- **RSI**: índice de força relativa

### Como explicar isso ao professor
Você pode dizer que este bloco transforma dados financeiros brutos em variáveis mais informativas para o aprendizado da rede neural.

In [5]:
def add_indicators(df: pd.DataFrame, col: str, window: int = 14) -> pd.DataFrame:
    out = df.copy()
    out[f'{col}_SMA_{window}'] = out[col].rolling(window=window).mean()
    out[f'{col}_EMA_{window}'] = out[col].ewm(span=window, adjust=False).mean()

    delta = out[col].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    out[f'{col}_RSI_{window}'] = 100 - (100 / (1 + rs))
    return out

## Bloco 6, Utilitários de coleta e organização dos datasets

### O que este bloco faz
Baixa os dados do Yahoo Finance para os 3 ativos escolhidos e monta um único dataframe multivariado.

### Intuito
Construir uma base única contendo:
- preço de fechamento de cada ticker
- indicadores técnicos de cada ticker

### Como explicar isso ao professor
Aqui, o modelo deixa de enxergar só um ativo isolado.
Ele passa a receber informações combinadas de múltiplos ativos, o que aumenta o contexto do problema.

In [6]:
def download_yfinance_data(tickers: List[str], start: str, end: str = None) -> pd.DataFrame:
    frames = []

    for ticker in tickers:
        df = yf.download(
            ticker,
            start=start,
            end=end,
            auto_adjust=True,
            progress=False,
            threads=False,
        )

        if df.empty:
            raise ValueError(f'Não foi possível baixar dados para {ticker}.')

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        base = df[['Close']].rename(columns={'Close': f'{ticker}_Close'})
        base = add_indicators(base, col=f'{ticker}_Close', window=14)
        frames.append(base)

    data = pd.concat(frames, axis=1).dropna().copy()
    data.index = pd.to_datetime(data.index)
    return data

## Bloco 7, Utilitários para séries temporais

### O que este bloco faz
Transforma os dados em sequências temporais.

### Intuito
Redes como LSTM e GRU não trabalham apenas com uma linha isolada.
Elas precisam de uma **janela de histórico** para aprender padrões ao longo do tempo.

### Como funciona
Se `SEQ_LEN = 60`, cada amostra de entrada contém os **60 dias anteriores**.
O alvo é o valor do dia seguinte que se quer prever.

### Como explicar isso ao professor
Este bloco converte a tabela tradicional em um formato adequado para redes neurais recorrentes.

In [7]:
def create_sequences(features: np.ndarray, target: np.ndarray, seq_len: int) -> Tuple[np.ndarray, np.ndarray]:
    X, y = [], []

    for i in range(seq_len, len(features)):
        X.append(features[i - seq_len:i])
        y.append(target[i])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

## Bloco 8, Classe de dataset para o PyTorch

### O que este bloco faz
Cria uma classe que organiza as sequências e os alvos no formato esperado pelo PyTorch.

### Intuito
Permitir o uso do `DataLoader`, que entrega os dados em lotes durante o treinamento.

### Como explicar isso ao professor
Este bloco é a ponte entre os arrays preparados e o pipeline de treinamento da biblioteca PyTorch.

In [8]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]

## Bloco 9, Definição dos modelos de rede neural

### O que este bloco faz
Define as duas arquiteturas que serão comparadas:
- **LSTM**
- **GRU**

### Intuito
Comparar duas abordagens recorrentes para o mesmo problema de previsão temporal.

### Como explicar isso ao professor
As duas redes são parecidas porque ambas foram criadas para lidar com dependência temporal.
A diferença é que a LSTM tem uma estrutura interna mais elaborada, enquanto a GRU é mais enxuta.

In [9]:
class LSTMModel(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 64, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)


class GRUModel(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 64, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.gru(x)
        out = out[:, -1, :]
        return self.fc(out)

## Bloco 10, Estrutura para guardar os resultados

### O que este bloco faz
Cria uma estrutura organizada para armazenar:
- nome do modelo
- histórico de treino
- métricas
- valores reais
- valores preditos
- próxima previsão

### Intuito
Deixar os resultados fáceis de consultar e comparar ao final.

In [10]:
@dataclass
class TrainingResult:
    name: str
    model: nn.Module
    history: Dict[str, List[float]]
    metrics: Dict[str, float]
    y_true: np.ndarray
    y_pred: np.ndarray
    next_pred: float

## Bloco 11, Utilitários de treinamento e previsão

### O que este bloco faz
Reúne as funções responsáveis por:
- treinar o modelo
- gerar predições no conjunto de teste
- estimar o próximo fechamento a partir da última janela disponível

### Intuito
Separar a lógica operacional do treinamento para deixar o notebook mais didático.

### Como explicar isso ao professor
Este bloco contém o ciclo principal do aprendizado supervisionado:
entrada, erro, ajuste dos pesos e validação.

In [11]:
def train_model(model: nn.Module, train_loader: DataLoader, test_loader: DataLoader, epochs: int, lr: float) -> Dict[str, List[float]]:
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'val_loss': []}

    model.to(DEVICE)

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)

                pred = model(X_batch)
                loss = criterion(pred, y_batch)
                val_losses.append(loss.item())

        history['train_loss'].append(float(np.mean(train_losses)))
        history['val_loss'].append(float(np.mean(val_losses)))

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"train_loss={history['train_loss'][-1]:.6f} | "
            f"val_loss={history['val_loss'][-1]:.6f}"
        )

    return history


def predict_model(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    preds = []

    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(DEVICE)
            pred = model(X_batch).cpu().numpy().flatten()
            preds.extend(pred)

    return np.array(preds)


def make_next_day_prediction(model: nn.Module, latest_window_scaled: np.ndarray, scaler_target: MinMaxScaler) -> float:
    model.eval()
    x = torch.tensor(latest_window_scaled[np.newaxis, :, :], dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        pred_scaled = model(x).cpu().numpy().flatten()[0]

    return float(inverse_target(np.array([pred_scaled]), scaler_target)[0])

## Bloco 12, Execução principal do pipeline

### O que este bloco faz
Executa o fluxo completo do projeto:
1. cria a pasta de saída
2. baixa os datasets
3. define colunas de entrada e alvo
4. separa treino e teste
5. normaliza os dados
6. cria sequências temporais
7. monta os `DataLoaders`
8. treina os modelos
9. calcula métricas
10. salva gráficos e arquivos CSV

### Intuito
Conectar todas as partes anteriores em um processo único e reprodutível.

### Como explicar isso ao professor
Este é o pipeline completo do experimento, do dado bruto até a comparação final entre LSTM e GRU.

In [12]:
def main() -> None:
    ensure_dir(OUTPUT_DIR)

    print('Baixando datasets do Yahoo Finance...')
    data = download_yfinance_data(TICKERS, START_DATE, END_DATE)
    data.to_csv(f'{OUTPUT_DIR}/dataset_multivariado.csv')

    feature_cols = list(data.columns)
    target_col = f'{TARGET_TICKER}_Close'

    split_idx = int(len(data) * TRAIN_SPLIT)
    train_df = data.iloc[:split_idx].copy()
    test_df = data.iloc[split_idx:].copy()

    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train_scaled = scaler_X.fit_transform(train_df[feature_cols])
    X_test_scaled = scaler_X.transform(test_df[feature_cols])

    y_train_scaled = scaler_y.fit_transform(train_df[[target_col]]).flatten()
    y_test_scaled = scaler_y.transform(test_df[[target_col]]).flatten()

    X_train, y_train = create_sequences(X_train_scaled, y_train_scaled, SEQ_LEN)

    combined_scaled = np.vstack([X_train_scaled[-SEQ_LEN:], X_test_scaled])
    combined_target = np.concatenate([y_train_scaled[-SEQ_LEN:], y_test_scaled])
    X_test, y_test = create_sequences(combined_scaled, combined_target, SEQ_LEN)

    train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

    input_size = X_train.shape[2]

    models = {
        'LSTM': LSTMModel(input_size=input_size),
        'GRU': GRUModel(input_size=input_size),
    }

    results: List[TrainingResult] = []

    for name, model in models.items():
        print(f'\nTreinando modelo: {name}')
        history = train_model(model, train_loader, test_loader, epochs=EPOCHS, lr=LEARNING_RATE)
        pred_scaled = predict_model(model, test_loader)

        y_true = inverse_target(y_test, scaler_y)
        y_pred = inverse_target(pred_scaled, scaler_y)
        metrics = evaluate_predictions(y_true, y_pred)
        next_pred = make_next_day_prediction(model, combined_scaled[-SEQ_LEN:], scaler_y)

        results.append(
            TrainingResult(
                name=name,
                model=model,
                history=history,
                metrics=metrics,
                y_true=y_true,
                y_pred=y_pred,
                next_pred=next_pred,
            )
        )

        pred_df = pd.DataFrame({
            'date': test_df.index,
            'real_close': y_true,
            'pred_close': y_pred,
        })
        pred_df.to_csv(f'{OUTPUT_DIR}/predicoes_{name.lower()}.csv', index=False)

        plt.figure(figsize=(12, 5))
        plt.plot(test_df.index, y_true, label='Real')
        plt.plot(test_df.index, y_pred, label=f'Predição {name}')
        plt.title(f'{name} | Real vs Predito | {TARGET_TICKER}')
        plt.xlabel('Data')
        plt.ylabel('Preço de fechamento')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/grafico_{name.lower()}.png', dpi=150)
        plt.close()

        plt.figure(figsize=(10, 4))
        plt.plot(history['train_loss'], label='Train loss')
        plt.plot(history['val_loss'], label='Validation loss')
        plt.title(f'{name} | Curva de treino')
        plt.xlabel('Época')
        plt.ylabel('Loss (MSE)')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/loss_{name.lower()}.png', dpi=150)
        plt.close()

    summary = []
    for r in results:
        summary.append({
            'modelo': r.name,
            'ticker_alvo': TARGET_TICKER,
            'datasets_usados': ', '.join(TICKERS),
            'seq_len': SEQ_LEN,
            'epocas': EPOCHS,
            'RMSE': r.metrics['RMSE'],
            'MAE': r.metrics['MAE'],
            'MAPE_%': r.metrics['MAPE_%'],
            'pred_proximo_fechamento': r.next_pred,
        })

    summary_df = pd.DataFrame(summary).sort_values('RMSE')
    summary_df.to_csv(f'{OUTPUT_DIR}/resumo_metricas.csv', index=False)

    print('\nResumo final:')
    print(summary_df.to_string(index=False))
    print(f'\nArquivos salvos em: {OUTPUT_DIR}')

## Bloco 13, Chamada da execução

### O que este bloco faz
Inicia de fato o experimento.

### Intuito
Rodar tudo de forma organizada a partir da função principal `main()`.

In [13]:
if __name__ == '__main__':
    main()

Baixando datasets do Yahoo Finance...

Treinando modelo: LSTM
Epoch 01/20 | train_loss=0.040388 | val_loss=0.021170
Epoch 02/20 | train_loss=0.001877 | val_loss=0.005901
Epoch 03/20 | train_loss=0.001013 | val_loss=0.008688
Epoch 04/20 | train_loss=0.000855 | val_loss=0.008920
Epoch 05/20 | train_loss=0.000807 | val_loss=0.006181
Epoch 06/20 | train_loss=0.000750 | val_loss=0.010540
Epoch 07/20 | train_loss=0.000839 | val_loss=0.006659
Epoch 08/20 | train_loss=0.000744 | val_loss=0.010598
Epoch 09/20 | train_loss=0.000607 | val_loss=0.005609
Epoch 10/20 | train_loss=0.000631 | val_loss=0.005131
Epoch 11/20 | train_loss=0.000632 | val_loss=0.005917
Epoch 12/20 | train_loss=0.000610 | val_loss=0.004527
Epoch 13/20 | train_loss=0.000608 | val_loss=0.005341
Epoch 14/20 | train_loss=0.000553 | val_loss=0.004956
Epoch 15/20 | train_loss=0.000593 | val_loss=0.004444
Epoch 16/20 | train_loss=0.000538 | val_loss=0.005599
Epoch 17/20 | train_loss=0.000551 | val_loss=0.004550
Epoch 18/20 | train_

## Resultado

LSTM → RMSE: 1.79, MAE: 1.22, MAPE: 3.68%, predição: 39.02

GRU → RMSE: 1.92, MAE: 1.13, MAPE: 3.30%, predição: 38.95

## Conclusão

O LSTM foi mais estável e melhor em evitar erros grandes (menor RMSE)

O GRU foi mais preciso no dia a dia (menor MAE e MAPE)

Os dois modelos tiveram desempenho muito próximo

Erro médio em torno de 3%, considerado bom para séries financeiras

## Conclusão:
Ambos funcionam bem para capturar tendência. O GRU tem leve vantagem em precisão média, enquanto o LSTM é mais robusto.